# EDGAR analysis — interactive demo

This notebook **runs the same offline fixture demo** as `python3 -m edgar_project.cli demo --fixtures`, then loads **unified findings** and a **suite summary** from disk.

**Requirements:** `pandas`, `matplotlib` (for one bar chart). The **Setup** code cell (imports + `REPO` path) runs next; it works from the repo root **or** from `notebooks/`.

Run the **Setup** cell below (imports, repository root, `PYTHONPATH`).

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Repository root (works if cwd is repo root OR notebooks/)
_here = Path.cwd().resolve()
if (_here / "edgar_project").is_dir():
    REPO = _here
elif (_here.parent / "edgar_project").is_dir():
    REPO = _here.parent
else:
    raise FileNotFoundError(
        "Could not find edgar_project/. Run Jupyter from the repo root or from notebooks/."
    )

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("Repository root:", REPO)

## 1. Run the fixture demo suite

**Pipeline (short):** ingest → quarterly panel → features → self/peer anomalies (+ optional trend-break) → **unified findings** table → Markdown report. **Orchestration** (live) chains MCP tools; this notebook only runs **fixtures** (no network).

This cell executes the analytical stack on small CSV fixtures (same as CLI `demo --fixtures`). Outputs go under `data/evaluation/suite_fixtures_v1/<case_id>/`.

In [ ]:
from edgar_project.evaluation.rubric import Rubric
from edgar_project.evaluation.runner import EvaluationRunner
from edgar_project.evaluation.schemas import BenchmarkSuite

suite_path = REPO / "edgar_project/evaluation/benchmarks/suite_fixtures_v1.json"
rubric_path = REPO / "edgar_project/evaluation/fixtures/rubric_baseline_v1.json"

suite = BenchmarkSuite.model_validate_json(suite_path.read_text(encoding="utf-8"))
rubric = Rubric.from_json_file(rubric_path)
runner = EvaluationRunner(suite=suite, rubric=rubric)
results = runner.run_suite()
summary = runner.latest_summary

print(f"Cases: {len(results)} | passed: {summary.passed_cases if summary else '—'}")
for r in results:
    print(f"  {r.case_id}: {r.status.value}")

## 2. Suite summary (JSON)

Aggregated pass/fail counts written next to per-case results. Paths are under `data/evaluation/`.

In [ ]:
out_dir = REPO / suite.output_dir
summary_path = out_dir / f"{suite.suite_id}_summary.json"
results_path = out_dir / f"{suite.suite_id}_results.json"

summary_blob = json.loads(summary_path.read_text(encoding="utf-8"))
display(pd.DataFrame([summary_blob]))

print("Summary file:", summary_path)
print("Results file: ", results_path)

## 3. Unified findings — peer outlier fixture

We load **`fixture_peer_relative_outlier`** — four synthetic peers where one name is an intentional outlier. The unified table is already sorted by severity (`score_adjusted`).

In [ ]:
payload = json.loads(results_path.read_text(encoding="utf-8"))
TARGET = "fixture_peer_relative_outlier"
row = next(r for r in payload["results"] if r["case_id"] == TARGET)
uf_path = Path(row["artifacts"]["unified_findings_csv"])

uf = pd.read_csv(uf_path)
print("Path:", uf_path)
display(uf[
    [
        "cik",
        "period",
        "metric",
        "finding_type",
        "direction",
        "score_adjusted",
        "overlap_count",
        "explanation_summary",
    ]
].head(8))

## 4. Unified findings — overlap (anomaly + trend-break)

**`fixture_anomaly_trend_overlap`** shows two sources on the same `(cik, period, metric)` when overlap metadata is present.

In [ ]:
TARGET2 = "fixture_anomaly_trend_overlap"
row2 = next(r for r in payload["results"] if r["case_id"] == TARGET2)
uf2 = pd.read_csv(row2["artifacts"]["unified_findings_csv"])
display(uf2[["cik", "period", "metric", "finding_source", "finding_type", "overlap_count", "overlap_sources"]])

## 5. Simple plot — finding types (peer scenario)

Single bar chart of `finding_type` counts — no seaborn or dashboards.

In [ ]:
counts = uf["finding_type"].astype(str).value_counts().sort_index()
ax = counts.plot(kind="bar", color="steelblue", figsize=(5, 2.8))
ax.set_title("Finding types — peer outlier fixture")
ax.set_ylabel("rows")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Optional: static samples only

If you did **not** run the suite, you can still inspect shape with repo examples (not live output):

- `examples/unified_findings.example.csv`
- `examples/report.example.md`

## Next steps

- **Live SEC + digest:** `python3 -m edgar_project.cli demo` (see root `README.md`).
- **Full artifact list:** re-run a case and open paths in `results` JSON under `artifacts`.